# Notebook 4: XGBoost Experiment
Trains, tunes, and evaluates the XGBoost classifier under in-distribution and cross-generator conditions.

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '..')
from src.features.classical import fit_transform_classical
from src.models.xgboost_model import train_xgboost, save_model
from src.evaluation.metrics import evaluate, cross_gen_drop

train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/val.csv')
test  = pd.read_csv('../data/processed/test_in_dist.csv')
test_cg = pd.read_csv('../data/processed/test_cross_gen.csv')

In [ ]:
features = fit_transform_classical(
    train['text'].tolist(), val['text'].tolist(),
    test['text'].tolist(),
)
X_train, X_val, X_test = features['train'], features['val'], features['test']
tfidf, stylo = features['tfidf'], features['stylo']

from scipy.sparse import hstack, csr_matrix
X_test_cg = hstack([tfidf.transform(test_cg['text']), csr_matrix(stylo.transform(test_cg['text']))])

In [ ]:
model = train_xgboost(X_train, train['label'])
save_model(model)

In [ ]:
# In-distribution evaluation
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
result_id = evaluate(test['label'], y_pred, y_prob, label='XGBoost (in-dist)')

In [ ]:
# Cross-generator evaluation
y_pred_cg = model.predict(X_test_cg)
y_prob_cg = model.predict_proba(X_test_cg)[:, 1]
result_cg = evaluate(test_cg['label'], y_pred_cg, y_prob_cg, label='XGBoost (cross-gen)')
drop = cross_gen_drop(result_id['f1'], result_cg['f1'])